In [1]:
#from tilke import Circuit
import pandas as pd
import numpy as np
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt
import time
import csv
import os

In [2]:

def read_custom_csv(file_path):
    """ Read a csv file and store in a dictionary each category"""

    data = {}
    current_title = None

    categories = ["EC-cones", "IC-cones", "F-cones", "EC-points", "IC-points","MC-points"]

    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip() # Remove non desired white spaces
            if line in categories:
                current_title = line
                data[current_title] = []
            else:
                x, y = map(float, line.split(',')) # Convert to float and separate by comma
                data[current_title].append((x, y))

    return data


# Function to get the midpoints given a triangle
def computeMidPoints(vertices1,vertices2, idx):
    
    # Compute midpoints between each vertex for both triangles
    mids1 = [((vertices1[i][0] + vertices1[(i+1)%3][0])/2, (vertices1[i][1] + vertices1[(i+1)%3][1])/2) for i in range(3)]
    mids2 = [((vertices2[i][0] + vertices2[(i+1)%3][0])/2, (vertices2[i][1] + vertices2[(i+1)%3][1])/2) for i in range(3)]

    # True points belong to the actual path
    truePoints = [mids1[0], mids1[2], mids2[1]] if idx==0 else [mids1[0],mids1[1],mids2[1]]
    # False point are middle points not contained in the path
    falsePoints = [mids1[1], mids2[0]] if idx==0 else [mids1[2],mids2[2]]

    return truePoints, falsePoints

def plotPoints(points,mark,plt):
    
    for point in points:
        plt.plot(point[0], point[1], mark)
    


# Function that gets midpoints from a triangulation
def get_midPoints(points, triangulation,midPoints,plt=None):
# p0 exterior left, p1 interior left, p2 exterior right, p3 interior right
    
    # Order indices of vertices
    sorted1 = np.sort(triangulation[0])
    sorted2 = np.sort(triangulation[1])

    # Ensure midpoint of p0-p1 is always first
    # If p0 in both triangles, sets are: sorted1=[p0,p1,p3], sorted2=[p0,p2,p3]
    # If p0 only in one trianlge: sets are: sorted1=[p0,p1,p2], sorted2=[p1,p2,p3]
    if not(sorted1[0] == 0 and sorted1[1] == 1):
        s = sorted1
        sorted1 = sorted2
        sorted2 = s


    # Given that triangles from triangulation give indices,
    # p0 == 0 (idx), p1 == 1, p2==2, p3==3 and so on
    # If idx = 0, triangle with diagonal p0-p3, else diagonal p1-p2
    idx = 0 if sorted2[0]==0 else 1
    true_midpoints, false_midpoints = computeMidPoints(points[sorted1], points[sorted2], idx)

    # Plot
    # plotPoints(midPoints["True"] + true_midpoints, 'rx', plt)
    # plotPoints(midPoints["False"] + false_midpoints, 'bx', plt)

    # Remove first element because it has been already stored in the previous computation
    true_midpoints = true_midpoints if len(midPoints["True"])==0 else true_midpoints[1:]
    # Update
    midPoints["True"] += true_midpoints
    midPoints["False"] += false_midpoints
    
    return midPoints


def add_to_csv(path,midPoints):
    # Add midpoint coordinates to csv
    with open(path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["True-MidPoints"])

        # Midpoints that belong to the path
        for midPoint in midPoints["True"]:
            writer.writerow(
                [
                    round(midPoint[0], 4), #x
                    round(midPoint[1], 4), #y
                ]
            )
        
        # Midpoints that do not belong to the path
       # writer.writerow(["False-MidPoints"])
       # for midpoint in midPoints["False"]:
       #     writer.writerow(
       #         [
       #             round(midPoint[0], 4), #x
       #             round(midPoint[1], 4), #y
       #         ]
       #     )

In [3]:
# Specify the path to CSV files within the subfolder
folder_path = './tilke_circuits_GT/'
# Get and store all files
file_list = os.listdir(folder_path)
csv_files = [file for file in file_list if file.endswith(".csv")]

In [4]:
# Keep track of progress
num_circuits_to_process = len(csv_files)
num_circuits_processed = 0

for csv_file in csv_files:
    csv_path = folder_path+csv_file
    data = read_custom_csv(csv_path)
    assert len(data["EC-cones"]) == len(data["IC-cones"]), "Number of cones at each curve should be the same"

    prev_cones = None
    # prev_point2 = np.array([data["EC-cones"][0], data["IC-cones"][0], data["EC-cones"][1], data["IC-cones"][1]])
    midPoints = {"True":[], "False":[]}

    for cones in zip(data["EC-cones"],data["IC-cones"]):
        if prev_cones is not None:
            # Get true midpoints
            # Compute triangulation every 4 points
            points = np.array([prev_cones[0], prev_cones[1], cones[0],cones[1]]) # 4 points
            triangulation = Delaunay(points)

            #plt.triplot(points[:, 0], points[:, 1], triangulation.simplices)
            #plt.plot(points[:, 0], points[:, 1], 'o')

            # Generate false midpoints
            # Compute triangulation for all points
            # prev_point2 = np.append(prev_point2,points,axis=0)
            # triangulation_add = Delaunay(prev_point2)
            # plt.triplot(prev_point2[:, 0], prev_point2[:, 1], triangulation_add.simplices)
            # plt.plot(prev_point2[:, 0], prev_point2[:, 1], 'o')
            midPoints = get_midPoints(points, triangulation.simplices,midPoints) #plt

            #plt.show()
            # time.sleep(5)

        prev_cones = [cones[0], cones[1]] # 2 points [x,y];[x,y]

    add_to_csv(csv_path, midPoints=midPoints)

    # Keep track of progress
    num_circuits_processed += 1
    print("Progress : ", 100 * num_circuits_processed/ num_circuits_to_process)

Progress :  1.0
Progress :  2.0
Progress :  3.0
Progress :  4.0
Progress :  5.0
Progress :  6.0
Progress :  7.0
Progress :  8.0
Progress :  9.0
Progress :  10.0
Progress :  11.0
Progress :  12.0
Progress :  13.0
Progress :  14.0
Progress :  15.0
Progress :  16.0
Progress :  17.0
Progress :  18.0
Progress :  19.0
Progress :  20.0
Progress :  21.0
Progress :  22.0
Progress :  23.0
Progress :  24.0
Progress :  25.0
Progress :  26.0
Progress :  27.0
Progress :  28.0
Progress :  29.0
Progress :  30.0
Progress :  31.0
Progress :  32.0
Progress :  33.0
Progress :  34.0
Progress :  35.0
Progress :  36.0
Progress :  37.0
Progress :  38.0
Progress :  39.0
Progress :  40.0
Progress :  41.0
Progress :  42.0
Progress :  43.0
Progress :  44.0
Progress :  45.0
Progress :  46.0
Progress :  47.0
Progress :  48.0
Progress :  49.0
Progress :  50.0
Progress :  51.0
Progress :  52.0
Progress :  53.0
Progress :  54.0
Progress :  55.0
Progress :  56.0
Progress :  57.0
Progress :  58.0
Progress :  59.0
Progre